> **Original course-project notebook (2026), kept for reference; superseded by `src/dfer/`.** Outputs are cleared (they contained DFEW frames, which cannot be redistributed). See the README for the final pipeline and results.

#Supervised ViT Baseline for DFER

In this notebook, we implement a supervised ViT (Vision Transformer) model as a baseline for the dynamic facial expression recognition (DFER) task. This model is pretrained on Kinetics-400, a large video dataset, and fine-tuned on DFEW for emotion classification.

We perform the following steps:

* Load the DFEW dataset for training and validation.
* Define the ViT architecture and its corresponding classifier.
* Train the model and evaluate its performance.
* Analyze the class distribution and handle class imbalance with class weights.

The final model will serve as a comparison point for our self-supervised models.

##Imports and Setup

We begin by importing the necessary libraries and setting up the directory paths for the project:

* Imports: Libraries for data processing, PyTorch operations, and visualization.
* Paths: We define the main project path (PROJECT_ROOT), source directory (SRC_DIR), and paths for configuration files and dataset directories.
* Output directories: Directories are created for storing logs, plots, and model checkpoints.

The configuration file (paths.yaml) is loaded to get paths for the DFEW dataset and train/test splits.

In [ ]:
# =========================
# 0) Imports + Paths
# =========================
import os, sys, time, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

from tqdm import tqdm
import yaml

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

CONFIG_PATH = PROJECT_ROOT / "configs" / "paths.yaml"
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

DFEW_ROOT   = config["dfew_root"]
SPLITS_ROOT = config["splits_root"]
print("DFEW_ROOT:", DFEW_ROOT)
print("SPLITS_ROOT:", SPLITS_ROOT)

# Outputs
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CKPT_DIR   = OUTPUT_DIR / "checkpoints"
LOG_DIR    = OUTPUT_DIR / "logs"
PLOT_DIR   = OUTPUT_DIR / "plots"
for d in [OUTPUT_DIR, CKPT_DIR, LOG_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)

##Reproducibility and Device Setup

In [ ]:
# =========================
# 1) Reproducibility + Device
# =========================
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

##Dataset Paths and Initialization

In [ ]:
# =========================
# 2) Fold + CSV paths
# =========================
FOLD = 1
TRAIN_CSV = os.path.join(SPLITS_ROOT, "train(single-labeled)", f"set_{FOLD}.csv")
VAL_CSV   = os.path.join(SPLITS_ROOT, "test(single-labeled)",  f"set_{FOLD}.csv")

print("TRAIN_CSV:", TRAIN_CSV)
print("VAL_CSV:", VAL_CSV)

RUN_NAME = f"02_supervised_vit_baseline_fold{FOLD}_FINAL"

##Data Loading and Class Weight Calculation

Here, we initialize the train and validation datasets for fold 1:

* The datasets are loaded using the DFEWVideoDataset class.
* Transformations (resize, normalize, tensor conversion) are applied to all images.
* We also define batch size, number of workers, and pin memory for the DataLoader to optimize training speed.

We then calculate the class weights based on the label distribution to handle class imbalance (with higher weights for less frequent classes).

In [ ]:
# =========================
# 3) Dataset + Transforms + Loaders
# =========================
from dataset import DFEWVideoDataset  # src/dataset.py

NUM_CLASSES = 7
NUM_FRAMES  = 16
IMG_SIZE    = 224

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

train_ds = DFEWVideoDataset(root_dir=DFEW_ROOT, split_csv=TRAIN_CSV, transform=train_transform, num_frames=NUM_FRAMES)
val_ds   = DFEWVideoDataset(root_dir=DFEW_ROOT, split_csv=VAL_CSV,   transform=val_transform,   num_frames=NUM_FRAMES)

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))

BATCH_SIZE  = 2
NUM_WORKERS = 4
PIN_MEMORY  = True

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

##Class Distribution and Class Weights

In this section, we:

* Read the CSV files containing the label information for the train and validation sets.
* Calculate the class distribution (number of samples per class) and visualize it with a plot.
* Calculate class weights for handling the class imbalance, where rarer classes are given higher weights.

The class distribution plot will help us understand if any classes are underrepresented.

In [ ]:
# =========================
# 4) Class distribution + class weights (imbalance)
# =========================
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

train_counts = train_df["label"].value_counts().sort_index()
val_counts   = val_df["label"].value_counts().sort_index()

dist = pd.DataFrame({
    "train_count": train_counts,
    "val_count": val_counts
}).fillna(0).astype(int)

print(dist)

dist_path = LOG_DIR / f"{RUN_NAME}_class_distribution.csv"
dist.to_csv(dist_path)
print("Saved:", dist_path)

plt.figure()
plt.plot(dist.index, dist["train_count"], label="train")
plt.plot(dist.index, dist["val_count"], label="val")
plt.xlabel("Class id (1..7)")
plt.ylabel("Count")
plt.title("Class distribution - " + RUN_NAME)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / f"{RUN_NAME}_class_distribution.png", dpi=200)
plt.show()

# Build class weights for labels 0..6
# CSV labels are 1..7, so we map them to 0..6 by sorting order
freq = train_counts.values.astype(np.float32)
weights = 1.0 / (freq + 1e-6)
weights = weights / weights.sum() * len(weights)

class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
print("Class weights (0..6):", class_weights)

##Supervised VideoMAE Model Setup

In this section, we load the pretrained VideoMAE model fine-tuned on Kinetics-400 and fine-tune it on the DFEW dataset for emotion classification. We use weighted cross-entropy loss to address class imbalance, and set up the AdamW optimizer with CosineAnnealingLR for learning rate scheduling. Mixed-precision training is enabled if a GPU is available to speed up the process.

In [ ]:
# =========================
# 6) Model: Supervised pretrained VideoMAE (Kinetics finetuned)
# =========================
from transformers import VideoMAEForVideoClassification

EPOCHS        = 20
LR            = 3e-5
WEIGHT_DECAY  = 0.05

MODEL_ID = "MCG-NJU/videomae-base-finetuned-kinetics"

model = VideoMAEForVideoClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
).to(device)

# FINAL: use weighted loss for imbalance fairness
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None

print("Loaded:", MODEL_ID)
print("Head reinitialized for 7 classes (expected).")

##Checkpoint and Resume Utilities

This section defines the utilities needed for saving and loading checkpoints during training:

* Checkpoint paths: where to save the best and last model weights, training history, and confusion matrices.
* Save/Load functions: These functions allow you to resume training from the last checkpoint or save the model’s state after each epoch.

We will also define paths to store logs, checkpoints, and plot images for future reference.

In [ ]:
# =========================
# 7) Checkpoint + Resume utilities
# =========================
BEST_ACC_PATH   = CKPT_DIR / f"{RUN_NAME}_best_acc.pth"
BEST_UAR_PATH   = CKPT_DIR / f"{RUN_NAME}_best_uar.pth"
LAST_FULL_PATH  = CKPT_DIR / f"{RUN_NAME}_last_full.pth"

HISTORY_CSV     = LOG_DIR / f"{RUN_NAME}_history.csv"
META_JSON       = LOG_DIR / f"{RUN_NAME}_meta.json"

CM_ACC_PATH     = PLOT_DIR / f"{RUN_NAME}_confusion_matrix_best_acc.png"
CM_UAR_PATH     = PLOT_DIR / f"{RUN_NAME}_confusion_matrix_best_uar.png"

def save_last_full(path, epoch, model, optimizer, scheduler, scaler,
                   best_val_acc, best_val_uar, history, early_state):
    ckpt = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_val_acc": best_val_acc,
        "best_val_uar": best_val_uar,
        "history": history,
        "early_state": early_state,
    }
    if scaler is not None:
        ckpt["scaler"] = scaler.state_dict()
    torch.save(ckpt, path)

def load_last_full(path, model, optimizer=None, scheduler=None, scaler=None):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model"])
    if optimizer is not None and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler is not None and "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])
    if scaler is not None and "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])
    return ckpt

##Training Configuration and Early Stopping

Here we define important hyperparameters and early stopping criteria for training:

* EPOCHS: Number of training epochs.
* LR: Learning rate.
* WEIGHT_DECAY: Regularization term.
* PATIENCE: Early stopping to avoid overfitting.

Early stopping ensures that training halts when no improvement in UAR is observed for a certain number of epochs.

In [ ]:
# =========================
# 8) Training config + Early stopping
# =========================
PATIENCE = 5
MIN_DELTA = 1e-4

history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
    "val_uar": [],
    "lr": [],
    "epoch_time_min": [],
}

best_val_acc = -1.0
best_val_uar = -1.0
start_epoch = 0

early_state = {"no_improve_uar": 0}

RESUME = False
if RESUME and LAST_FULL_PATH.exists():
    ckpt = load_last_full(LAST_FULL_PATH, model, optimizer, scheduler, scaler)
    start_epoch = ckpt["epoch"] + 1
    best_val_acc = ckpt.get("best_val_acc", -1.0)
    best_val_uar = ckpt.get("best_val_uar", -1.0)
    history = ckpt.get("history", history)
    early_state = ckpt.get("early_state", early_state)
    print(f"Resumed from {LAST_FULL_PATH} at epoch {start_epoch}, best_acc={best_val_acc:.4f}, best_uar={best_val_uar:.4f}")
else:
    print("Starting fresh training.")

##Training and Validation Loops

This section contains the training and validation loops for each epoch. During training, we update the model’s weights based on the loss. In the validation loop, we evaluate the model’s performance on the validation set.

The model is trained with gradient scaling to use mixed precision if a GPU is available, which speeds up training without losing model precision.

In [ ]:
# =========================
# 9) Train + Val loops
# =========================
def to_videomae_format(videos, num_frames=16):
    """
    Ensure (B,T,C,H,W) for HF VideoMAE.
    Dataset returns (C,T,H,W).
    Dataloader gives (B,C,T,H,W).
    """
    if videos.dim() == 5 and videos.shape[1] == 3 and videos.shape[2] == num_frames:
        return videos.permute(0, 2, 1, 3, 4)  # (B,T,C,H,W)
    return videos

total_start = time.time()

for epoch in range(start_epoch, EPOCHS):
    epoch_start = time.time()

    # ---------- TRAIN ----------
    model.train()
    train_losses, train_preds, train_targets = [], [], []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=True)
    for videos, labels in pbar:
        videos = to_videomae_format(videos, NUM_FRAMES)
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if device == "cuda":
            with torch.amp.autocast("cuda"):
                logits = model(videos).logits
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(videos).logits
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

        train_losses.append(loss.item())
        preds = logits.argmax(1).detach().cpu().numpy()
        train_preds.extend(preds)
        train_targets.extend(labels.detach().cpu().numpy())

        running_acc = accuracy_score(train_targets, train_preds)
        pbar.set_postfix({
            "loss": float(np.mean(train_losses)),
            "acc": float(running_acc),
            "lr": float(optimizer.param_groups[0]["lr"])
        })

    train_loss = float(np.mean(train_losses))
    train_acc  = float(accuracy_score(train_targets, train_preds))

    # ---------- VAL ----------
    model.eval()
    val_losses, val_preds, val_targets = [], [], []

    vbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False)
    with torch.no_grad():
        for videos, labels in vbar:
            videos = to_videomae_format(videos, NUM_FRAMES)
            videos = videos.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(videos).logits
            loss = criterion(logits, labels)

            val_losses.append(loss.item())
            val_preds.extend(logits.argmax(1).cpu().numpy())
            val_targets.extend(labels.cpu().numpy())

    val_loss = float(np.mean(val_losses))
    val_acc  = float(accuracy_score(val_targets, val_preds))
    val_uar  = float(recall_score(val_targets, val_preds, average="macro"))

    scheduler.step()
    lr_now = float(optimizer.param_groups[0]["lr"])
    epoch_time = (time.time() - epoch_start) / 60.0

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss {train_loss:.4f} | Acc {train_acc:.4f}")
    print(f"Val   Loss {val_loss:.4f} | Acc {val_acc:.4f} | UAR {val_uar:.4f}")
    print(f"LR {lr_now:.6f} | Time {epoch_time:.2f} min")

    # ---------- log history ----------
    history["epoch"].append(epoch+1)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_uar"].append(val_uar)
    history["lr"].append(lr_now)
    history["epoch_time_min"].append(epoch_time)

    pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)

    # ---------- save LAST (resume) ----------
    save_last_full(LAST_FULL_PATH, epoch, model, optimizer, scheduler, scaler,
                   best_val_acc, best_val_uar, history, early_state)

    # ---------- save BEST by ACC ----------
    if val_acc > best_val_acc + MIN_DELTA:
        best_val_acc = val_acc
        torch.save(model.state_dict(), BEST_ACC_PATH)
        print("New BEST-ACC saved.")

    # ---------- save BEST by UAR + early stopping ----------
    if val_uar > best_val_uar + MIN_DELTA:
        best_val_uar = val_uar
        torch.save(model.state_dict(), BEST_UAR_PATH)
        early_state["no_improve_uar"] = 0
        print("New BEST-UAR saved.")
    else:
        early_state["no_improve_uar"] += 1
        print(f"No UAR improvement {early_state['no_improve_uar']}/{PATIENCE}")

    if early_state["no_improve_uar"] >= PATIENCE:
        print("Early stopping (UAR) triggered.")
        break

total_time = (time.time() - total_start) / 60.0
print(f"\nTraining finished. Total time: {total_time:.2f} min")
print(f"Best Val Acc: {best_val_acc:.4f}")
print(f"Best Val UAR: {best_val_uar:.4f}")

##Saving Checkpoints and Confusion Matrices

This section saves the model’s checkpoint after each epoch and evaluates its performance by saving confusion matrices for both best accuracy and best UAR.

In [ ]:
# =========================
# 10) Save meta
# =========================
meta = {
    "run_name": RUN_NAME,
    "fold": FOLD,
    "seed": SEED,
    "epochs_requested": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "num_frames": NUM_FRAMES,
    "img_size": IMG_SIZE,
    "model_id": MODEL_ID,
    "patience": PATIENCE,
    "min_delta": MIN_DELTA,
    "primary_metric": "val_uar",
    "weighted_loss": True,
    "train_aug": "RandomResizedCrop(0.8-1.0)+HFlip",
    "val_aug": "Resize",
}
with open(META_JSON, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved:", HISTORY_CSV)
print("Saved:", META_JSON)
print("Best weights acc:", BEST_ACC_PATH)
print("Best weights uar:", BEST_UAR_PATH)
print("Last full ckpt:", LAST_FULL_PATH)

In [ ]:
# =========================
# 11) Plots (saved + shown)
# =========================
hist = pd.read_csv(HISTORY_CSV)

plt.figure()
plt.plot(hist["epoch"], hist["train_loss"], label="train_loss")
plt.plot(hist["epoch"], hist["val_loss"], label="val_loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.grid(True)
plt.title(RUN_NAME + " loss")
plt.tight_layout()
plt.savefig(PLOT_DIR / f"{RUN_NAME}_loss.png", dpi=200)
plt.show()

plt.figure()
plt.plot(hist["epoch"], hist["train_acc"], label="train_acc")
plt.plot(hist["epoch"], hist["val_acc"], label="val_acc")
plt.xlabel("epoch"); plt.ylabel("acc"); plt.legend(); plt.grid(True)
plt.title(RUN_NAME + " accuracy")
plt.tight_layout()
plt.savefig(PLOT_DIR / f"{RUN_NAME}_accuracy.png", dpi=200)
plt.show()

plt.figure()
plt.plot(hist["epoch"], hist["val_uar"], label="val_uar")
plt.xlabel("epoch"); plt.ylabel("UAR"); plt.legend(); plt.grid(True)
plt.title(RUN_NAME + " UAR")
plt.tight_layout()
plt.savefig(PLOT_DIR / f"{RUN_NAME}_uar.png", dpi=200)
plt.show()

print("Saved plots to:", PLOT_DIR)

In [ ]:
# =========================
# 12) Confusion matrices for BEST-ACC and BEST-UAR
# =========================
def eval_and_save_cm(weights_path, cm_path, title):
    if not weights_path.exists():
        raise FileNotFoundError(f"Missing weights: {weights_path}")

    model.load_state_dict(torch.load(weights_path, map_location="cpu"))
    model.to(device)
    model.eval()

    all_preds, all_targets = [], []
    with torch.no_grad():
        for videos, labels in tqdm(val_loader, desc=f"Eval {title}", leave=False):
            videos = to_videomae_format(videos, NUM_FRAMES)
            videos = videos.to(device)
            logits = model(videos).logits
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_targets.extend(labels.numpy())

    cm = confusion_matrix(all_targets, all_preds)

    plt.figure(figsize=(6,6))
    plt.imshow(cm)
    plt.title(title)
    plt.xlabel("Pred"); plt.ylabel("True")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(cm_path, dpi=200)
    plt.show()

    final_acc = accuracy_score(all_targets, all_preds)
    final_uar = recall_score(all_targets, all_preds, average="macro")
    print(f"{title}: Acc={final_acc:.4f} | UAR={final_uar:.4f}")
    print("Saved:", cm_path)

eval_and_save_cm(BEST_ACC_PATH, CM_ACC_PATH, f"Confusion Matrix (BEST ACC) - {RUN_NAME}")
eval_and_save_cm(BEST_UAR_PATH, CM_UAR_PATH, f"Confusion Matrix (BEST UAR) - {RUN_NAME}")